In [12]:
import pandas as pd
import numpy as np
import re
from collections import Counter
misclassified_df = pd.read_csv("baseline_misclassified_development.csv")
df_all = pd.read_csv("../data/raw/development.csv")

In [6]:
df = misclassified_df.copy()   # <-- sostituisci se il nome è diverso


In [7]:
df["err_pair"] = df["y_true"].astype(str) + "→" + df["y_pred"].astype(str)
df["err_pair"].value_counts().head(15)

err_pair
0→5    3534
5→0    2450
5→3    1566
0→3    1413
3→5    1389
3→0     887
0→1     837
5→1     705
3→4     668
5→4     650
2→1     628
3→2     554
3→1     548
1→2     538
0→2     508
Name: count, dtype: int64

In [8]:
df["y_true"].value_counts(normalize=True).sort_index()


y_true
0    0.310039
1    0.087900
2    0.081938
3    0.191539
4    0.032179
5    0.268040
6    0.028365
Name: proportion, dtype: float64

In [9]:
df["y_true"].value_counts(normalize=True).sort_index()

y_true
0    0.310039
1    0.087900
2    0.081938
3    0.191539
4    0.032179
5    0.268040
6    0.028365
Name: proportion, dtype: float64

In [10]:
df["y_pred"].value_counts(normalize=True).sort_index()


y_pred
0    0.188558
1    0.125208
2    0.086892
3    0.187856
4    0.077598
5    0.261245
6    0.072644
Name: proportion, dtype: float64

In [13]:
err_rate = (
    df.groupby("source").size()
    / df_all.groupby("source").size()
).dropna().sort_values(ascending=False)

err_rate.head(15)


source
romanticmovies.about.com    1.0
Lucianne                    1.0
MaineToday.com              1.0
Maltamedia                  1.0
Marshalltown                1.0
Matamat.com                 1.0
MediaGab                    1.0
MegaStar.co.uk              1.0
Melrose                     1.0
Memphis                     1.0
Albawaba                    1.0
Aljazeera.com               1.0
AngolaPress                 1.0
Accountingweb.com           1.0
Accra                       1.0
dtype: float64

In [15]:
df[["n_tokens","article_len","title_len","title_ratio"]].describe(percentiles=[0.1,0.25,0.5,0.75,0.9])



,n_tokens,article_len,title_len,title_ratio
count,22810.000000,22810.000000,22810.000000,22810.000000
mean,29.098641,182.071021,41.756905,0.290727
std,11.307766,88.783731,12.154263,0.736456
min,0.000000,0.000000,0.000000,0.000000
10%,17.000000,103.000000,29.000000,0.146226
25%,22.000000,136.000000,33.000000,0.189474
50%,29.000000,178.000000,41.000000,0.241860
75%,35.000000,215.000000,49.000000,0.302752
90%,39.000000,241.000000,58.000000,0.357678
max,216.000000,1437.000000,135.000000,22.333333


In [16]:
df[["n_tokens","article_len","title_len","title_ratio"]].describe(percentiles=[0.1,0.25,0.5,0.75,0.9])


,n_tokens,article_len,title_len,title_ratio
count,22810.000000,22810.000000,22810.000000,22810.000000
mean,29.098641,182.071021,41.756905,0.290727
std,11.307766,88.783731,12.154263,0.736456
min,0.000000,0.000000,0.000000,0.000000
10%,17.000000,103.000000,29.000000,0.146226
25%,22.000000,136.000000,33.000000,0.189474
50%,29.000000,178.000000,41.000000,0.241860
75%,35.000000,215.000000,49.000000,0.302752
90%,39.000000,241.000000,58.000000,0.357678
max,216.000000,1437.000000,135.000000,22.333333


In [17]:
def tokenize(text):
    return re.findall(r"[a-z0-9_:/\-]+", text.lower())


In [18]:
err_tokens = Counter()
for t in df["text"]:
    err_tokens.update(tokenize(t))

err_tokens.most_common(50)


[('the', 35890),
 ('a', 21895),
 ('to', 21249),
 ('of', 19551),
 ('in', 18766),
 ('s', 11996),
 ('and', 11967),
 ('on', 11202),
 ('for', 8869),
 ('39', 7332),
 ('-', 7138),
 ('that', 5014),
 ('reuters', 4658),
 ('at', 4621),
 ('is', 4559),
 ('with', 4507),
 ('as', 4477),
 ('by', 3911),
 ('new', 3772),
 ('said', 3746),
 ('an', 3495),
 ('from', 3392),
 ('it', 3288),
 ('has', 3255),
 ('after', 3037),
 ('his', 2917),
 ('was', 2876),
 ('its', 2875),
 ('us', 2780),
 ('will', 2250),
 ('be', 2244),
 ('are', 2209),
 ('have', 2180),
 ('quot', 1911),
 ('over', 1878),
 ('u', 1781),
 ('he', 1766),
 ('more', 1754),
 ('their', 1719),
 ('ap', 1664),
 ('but', 1655),
 ('iraq', 1643),
 ('who', 1606),
 ('two', 1593),
 ('president', 1556),
 ('up', 1521),
 ('not', 1466),
 ('people', 1434),
 ('says', 1420),
 ('world', 1412)]

In [ ]:

ok_tokens = Counter()
for t in ok_df["article"]:
    ok_tokens.update(tokenize(t))


AttributeError: 'float' object has no attribute 'lower'

In [22]:
delta = {}

for tok, c_err in err_tokens.items():
    c_ok = ok_tokens.get(tok, 0)
    if c_err >= 50:
        delta[tok] = c_err / (c_ok + 1)

pd.Series(delta).sort_values(ascending=False).head(30)


football:    42.666667
kills        11.347826
sues          7.500000
rejects       6.750000
urges         6.300000
vows          6.300000
/b            6.000000
ny            5.250000
fda           5.125000
quake         5.100000
dies          5.025641
unveils       5.000000
backs         4.470588
seeks         4.235294
raises        3.809524
confirms      3.476190
pm            3.448276
wins          3.276316
agrees        3.217391
crashes       3.125000
loses         3.105263
denies        3.000000
celebrex      2.956522
sees          2.951220
blasts        2.925926
cow           2.857143
blast         2.809524
pill          2.782609
mad           2.781250
fails         2.772727
dtype: float64

In [23]:
def extract_tags(text):
    return re.findall(r"<\s*([a-z0-9]+)", text.lower())


In [ ]:
tag_counter = Counter()

for t in df["article"]:

tag_counter.most_common(30)


AttributeError: 'float' object has no attribute 'lower'

In [29]:
import pandas as pd
import numpy as np
import re
from collections import Counter

# ============================================================
# SETUP
# ============================================================

df = pd.read_csv("baseline_misclassified_development.csv")

df = df.copy()
df_err = df[df["y_true"] != df["y_pred"]].copy()

print("Total samples:", len(df))
print("Misclassified:", len(df_err))
print("Error rate:", len(df_err) / len(df))

# ============================================================
# 1. ERROR PAIRS (y_true → y_pred)
# ============================================================

df_err["err_pair"] = df_err["y_true"].astype(str) + "→" + df_err["y_pred"].astype(str)

print("\n=== TOP ERROR PAIRS ===")
print(df_err["err_pair"].value_counts().head(20))

# ============================================================
# 2. LABEL-WISE ERROR ANALYSIS
# ============================================================

print("\n=== ERROR DISTRIBUTION | TRUE LABEL ===")
print(df_err["y_true"].value_counts(normalize=True).sort_index())

print("\n=== P(ERROR | TRUE LABEL) ===")
print(
	df_err["y_true"].value_counts()
	/ df["y_true"].value_counts()
)

# ============================================================
# 3. SOURCE ANALYSIS
# ============================================================

print("\n=== TOP SOURCES IN ERRORS ===")
print(df_err["source"].value_counts().head(20))

print("\n=== ERROR RATE BY SOURCE ===")
err_rate_by_source = (
	df_err.groupby("source").size()
	/ df.groupby("source").size()
).dropna().sort_values(ascending=False)

print(err_rate_by_source.head(20))

# ============================================================
# 4. LENGTH / STRUCTURE ANALYSIS
# ============================================================

num_cols = ["n_tokens", "article_len", "title_len", "title_ratio"]

print("\n=== LENGTH STATS | ERRORS ===")
print(df_err[num_cols].describe(percentiles=[0.1,0.25,0.5,0.75,0.9]))

print("\n=== LENGTH STATS | CORRECT ===")
print(df[df["y_true"] == df["y_pred"]][num_cols]
	  .describe(percentiles=[0.1,0.25,0.5,0.75,0.9]))

# ============================================================
# 5. TOKEN ANALYSIS (ERROR-SPECIFIC)
# ============================================================

def tokenize(text):
	return re.findall(r"[a-z0-9_:/\-]+", str(text).lower())

err_tokens = Counter()
for t in df_err["text"]:
	err_tokens.update(tokenize(t))

print("\n=== TOP TOKENS IN ERRORS ===")
print(err_tokens.most_common(50))

ok_tokens = Counter()
for t in df[df["y_true"] == df["y_pred"]]["text"]:
	ok_tokens.update(tokenize(t))

token_ratio = {}
for tok, c_err in err_tokens.items():
	if c_err >= 50:
		token_ratio[tok] = c_err / (ok_tokens.get(tok, 0) + 1)

print("\n=== TOKENS OVER-REPRESENTED IN ERRORS ===")
print(pd.Series(token_ratio).sort_values(ascending=False).head(30))

# ============================================================
# 6. HTML / META TAG ANALYSIS
# ============================================================

def extract_tags(text):
	return re.findall(r"<\s*([a-z0-9]+)", str(text).lower())

err_tags = Counter()
for t in df_err["article"]:
	err_tags.update(extract_tags(t))

all_tags = Counter()
for t in df["article"]:
	all_tags.update(extract_tags(t))

tag_ratio = {}
for tag, c_err in err_tags.items():
	if c_err >= 20:
		tag_ratio[tag] = c_err / (all_tags.get(tag, 0) + 1)

print("\n=== TOP HTML TAGS IN ERRORS ===")
print(err_tags.most_common(30))

print("\n=== HTML TAGS OVER-REPRESENTED IN ERRORS ===")
print(pd.Series(tag_ratio).sort_values(ascending=False).head(20))

# ============================================================
# 7. TIMESTAMP ANALYSIS
# ============================================================

ts_missing = df["timestamp"] == "0000-00-00 00:00:00"

print("\nMissing timestamp ratio (ALL):", ts_missing.mean())
print("Missing timestamp ratio (ERRORS):",
	  (df_err["timestamp"] == "0000-00-00 00:00:00").mean())

print("\n=== ERROR RATE | TIMESTAMP MISSING ===")
print(
	df_err[ts_missing].shape[0] /
	df[ts_missing].shape[0]
)

# ============================================================
# 8. DUPLICATES & CONFLICTS
# ============================================================

dup_text = df_err[df_err.duplicated("text", keep=False)]

print("\n=== DUPLICATE TEXTS IN ERRORS ===")
print("Duplicate rows:", len(dup_text))
print("Duplicate ratio:", len(dup_text) / len(df_err))

label_conflict = (
	df_err.groupby("text")["y_true"]
	.nunique()
	.reset_index()
)

print("\n=== DUPLICATES WITH MULTIPLE TRUE LABELS ===")
print(label_conflict[label_conflict["y_true"] > 1].head())

print("\n=== DUPLICATES ACROSS FOLDS ===")
print(df_err.groupby("text")["fold"].nunique().value_counts())

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

summary = {
	"n_total": len(df),
	"n_errors": len(df_err),
	"error_rate": len(df_err) / len(df),
	"top_error_pairs": df_err["err_pair"].value_counts().head(5).to_dict(),
	"top_sources": df_err["source"].value_counts().head(5).to_dict(),
	"duplicate_ratio": len(dup_text) / len(df_err),
}

print("\n=== FINAL SUMMARY ===")
print(summary)



Total samples: 22810
Misclassified: 22810
Error rate: 1.0

=== TOP ERROR PAIRS ===
err_pair
0→5    3534
5→0    2450
5→3    1566
0→3    1413
3→5    1389
3→0     887
0→1     837
5→1     705
3→4     668
5→4     650
2→1     628
3→2     554
3→1     548
1→2     538
0→2     508
0→6     496
5→6     425
1→3     411
1→5     397
2→3     389
Name: count, dtype: int64

=== ERROR DISTRIBUTION | TRUE LABEL ===
y_true
0    0.310039
1    0.087900
2    0.081938
3    0.191539
4    0.032179
5    0.268040
6    0.028365
Name: proportion, dtype: float64

=== P(ERROR | TRUE LABEL) ===
y_true
0    1.0
5    1.0
3    1.0
1    1.0
2    1.0
4    1.0
6    1.0
Name: count, dtype: float64

=== TOP SOURCES IN ERRORS ===
source
Reuters          4200
BBC              4144
New              1938
Yahoo            1564
Washington        849
RedNova           396
ABC               341
Boston            328
Xinhua            305
Guardian          289
CNN               228
San               213
Bloomberg         189
Seattle   

In [33]:
import pandas as pd
import numpy as np

# ============================================================
# LOAD
# ============================================================
df_dev = pd.read_csv("../data/raw/development.csv")
df_eval = pd.read_csv("../data/raw/evaluation.csv")
df = df_dev.copy()

print("Total samples:", len(df))
def safe_unique(x):
	return sorted(v for v in x.unique() if pd.notna(v))
# ============================================================
# 1. DUPLICATI PER ARTICLE (testo identico)
# ============================================================


dup_article = df[df.duplicated("article", keep=False)].copy()

print("\n=== DUPLICATES BY ARTICLE ===")
print("Rows:", len(dup_article))
print("Unique articles:", dup_article["article"].nunique())

article_conflicts = (
	dup_article
	.groupby("article")
	.agg(
		n_labels=("label", "nunique"),
		labels=("label", safe_unique),
		sources=("source", safe_unique),
		titles=("title", safe_unique),
		timestamps=("timestamp", safe_unique)
	)
	.reset_index()
)

conflict_articles = article_conflicts[article_conflicts["n_labels"] > 1]

print("\n=== ARTICLE DUPLICATES WITH LABEL CONFLICT ===")
print("Conflicting articles:", len(conflict_articles))
print(conflict_articles.head(10))

# ============================================================
# 2. COSA CAMBIA DAVVERO?
# ============================================================

def diff_summary(df_conf):
	return pd.Series({
		"diff_source": df_conf["sources"].apply(len).gt(1).mean(),
		"diff_title": df_conf["titles"].apply(len).gt(1).mean(),
		"diff_timestamp": df_conf["timestamps"].apply(len).gt(1).mean()
	})

print("\n=== WHAT DIFFERS IN ARTICLE CONFLICTS ===")
print(diff_summary(conflict_articles))

# ============================================================
# 3. DUPLICATI PER TITLE
# ============================================================

dup_title = df[df.duplicated("title", keep=False)].copy()

print("\n=== DUPLICATES BY TITLE ===")
print("Rows:", len(dup_title))
print("Unique titles:", dup_title["title"].nunique())

title_conflicts = (
	dup_title.groupby("title")
	.agg(
		n_labels=("label", "nunique"),
		labels=("label", lambda x: sorted(x.unique())),
		sources=("source", lambda x: sorted(x.unique())),
		articles=("article", lambda x: len(x.unique()))
	)
	.reset_index()
)

conflict_titles = title_conflicts[title_conflicts["n_labels"] > 1]

print("\n=== TITLE DUPLICATES WITH LABEL CONFLICT ===")
print("Conflicting titles:", len(conflict_titles))
print(conflict_titles.head(10))

print("\n=== TITLE CONFLICT SUMMARY ===")

summary_df = pd.DataFrame({
	"n_articles": conflict_titles["articles"],
	"n_sources": conflict_titles["sources"].apply(len)
})

print(summary_df.describe())

# ============================================================
# 4. POTENTIAL DROP IMPACT
# ============================================================

drop_articles = conflict_articles["article"]
drop_titles   = conflict_titles["title"]

drop_mask = df["article"].isin(drop_articles) | df["title"].isin(drop_titles)

print("\n=== DROP IMPACT ESTIMATE ===")
print("Rows to drop:", drop_mask.sum())
print("Drop ratio:", drop_mask.mean())

print("\nLabel distribution BEFORE drop:")
print(df["label"].value_counts(normalize=True).sort_index())

print("\nLabel distribution AFTER drop:")
print(df[~drop_mask]["label"].value_counts(normalize=True).sort_index())

# ============================================================
# 5. BACKCHECK ON EVALUATION
# ============================================================

# df_eval = pd.read_csv("evaluation.csv")

common_articles = set(df["article"]) & set(df_eval["article"])
common_titles   = set(df["title"]) & set(df_eval["title"])

print("\n=== BACKCHECK EVAL ===")
print("Common articles DEV ∩ EVAL:", len(common_articles))
print("Common titles DEV ∩ EVAL:", len(common_titles))

# ============================================================
# 6. FINAL COUNTS
# ============================================================

summary = {
	"total_dev": len(df),
	"dup_articles": len(dup_article),
	"conflict_articles": len(conflict_articles),
	"dup_titles": len(dup_title),
	"conflict_titles": len(conflict_titles),
	"drop_ratio_est": drop_mask.mean(),
	"common_articles_eval": len(common_articles),
	"common_titles_eval": len(common_titles)
}

print("\n=== SUMMARY ===")
for k, v in summary.items():
	print(f"{k}: {v}")


Total samples: 79997

=== DUPLICATES BY ARTICLE ===
Rows: 8426
Unique articles: 2823

=== ARTICLE DUPLICATES WITH LABEL CONFLICT ===
Conflicting articles: 1614
                                              article  n_labels     labels  \
1                                                             3  [0, 2, 4]   
2    A start-up founded by a former telecom regula...         2     [0, 2]   
3    ABIDJAN (Reuters) - Hundreds of French citize...         2     [0, 5]   
4    ABUJA (Reuters) - A transport mix-up delayed ...         2     [0, 5]   
5    AGADEZ, Niger (Reuters) - Customs officers in...         2     [0, 3]   
7    ATHENS (Reuters) - The United States crashed ...         2     [4, 5]   
8    BAGHDAD (Reuters) - Gunmen killed Baghdad's g...         2     [0, 5]   
9    BAGHDAD (Reuters) - Hundreds of Iraqi troops ...         2     [0, 5]   
10   BAGHDAD (Reuters) - Iraq's plans to hold elec...         2     [0, 5]   
11   BAGHDAD (Reuters) - Iraq's president forecast...       

In [ ]:
import pandas as pd
import numpy as np

# ================================
# 1. DUPLICATED ARTICLES
# ================================
dup_df = df[df.duplicated("article", keep=False)].copy()

print("Total duplicated rows:", len(dup_df))
print("Unique duplicated articles:", dup_df["article"].nunique())

# ================================
#  KEEP ONLY LABEL-CONFLICT ARTICLES
# ================================
conflict_articles = (
	dup_df.groupby("article")["y_true"]
	.nunique()
	.reset_index()
	.query("y_true > 1")["article"]
)

conflict_df = dup_df[dup_df["article"].isin(conflict_articles)].copy()

print("Conflicting duplicated articles:", conflict_df["article"].nunique())
print("Rows involved:", len(conflict_df))

# ================================
# 3. ANALYZE PREDICTION COLLAPSE
# ================================
def majority(series):
	return series.value_counts().idxmax()

group_analysis = (
	conflict_df
	.groupby("article")
	.apply(lambda g: pd.Series({
		"n_samples": len(g),
		"true_labels": sorted(g["y_true"].unique()),
		"true_majority": majority(g["y_true"]),
		"pred_majority": majority(g["y_pred"]),
		"pred_entropy": -np.sum(
			(g["y_pred"].value_counts(normalize=True) *
			 np.log2(g["y_pred"].value_counts(normalize=True) + 1e-12))
		),
		"agreement_with_true_majority":
			majority(g["y_pred"]) == majority(g["y_true"])
	}))
	.reset_index()
)

print("\n=== DUPLICATE COLLAPSE SUMMARY ===")
print(group_analysis.head())

# ================================
# 4. GLOBAL METRICS
# ================================
collapse_rate = (
	group_analysis["pred_majority"]
	.value_counts(normalize=True)
	.sort_index()
)

agreement_rate = group_analysis["agreement_with_true_majority"].mean()

print("\n=== COLLAPSE DISTRIBUTION (pred_majority) ===")
print(collapse_rate)

print("\nAgreement with true majority:", round(agreement_rate, 4))

# ================================
# 5. CONFUSION ON DUPLICATES ONLY
# ================================
dup_confusion = (
	conflict_df
	.groupby(["y_true", "y_pred"])
	.size()
	.reset_index(name="count")
	.sort_values("count", ascending=False)
)

print("\n=== TOP CONFUSIONS ON DUPLICATES ===")
print(dup_confusion.head(15))

# ================================
# 6. OPTIONAL: SOURCE EFFECT
# ================================
source_bias = (
	conflict_df
	.groupby("source")["y_pred"]
	.value_counts(normalize=True)
	.unstack(fill_value=0)
)

print("\n=== SOURCE → PRED DISTRIBUTION (DUPLICATES) ===")
print(source_bias.head())


Total duplicated rows: 8426
Unique duplicated articles: 2823


KeyError: 'Column not found: y_true'

In [35]:
import pandas as pd
import numpy as np

# ================================
# 0. PREP
# ================================
df = df.copy()

df["ts_valid"] = (
	df["timestamp"].notna() &
	(df["timestamp"] != "0000-00-00 00:00:00")
)

df["timestamp_parsed"] = pd.to_datetime(
	df["timestamp"],
	errors="coerce"
)

# ================================
# 1. KEEP DUPLICATED ARTICLES
# ================================
dup_df = df[df.duplicated("article", keep=False)].copy()

print("Total duplicated rows:", len(dup_df))
print("Unique duplicated articles:", dup_df["article"].nunique())

# ================================
# 2. ARTICLE-LEVEL ANALYSIS
# ================================
def analyze_article(g):
	g = g.sort_values("timestamp_parsed")

	valid_ts = g[g["ts_valid"]]

	first_label = (
		valid_ts.iloc[0]["label"]
		if len(valid_ts) > 0 else None
	)

	return pd.Series({
		"n_rows": len(g),
		"n_labels": g["label"].nunique(),
		"labels": sorted(g["label"].unique()),
		"has_valid_ts": len(valid_ts) > 0,
		"first_ts_label": first_label,
		"majority_label": g["label"].value_counts().idxmax(),
		"agree_first_majority":
			first_label == g["label"].value_counts().idxmax()
			if first_label is not None else np.nan,
		"p_same_as_first_ts":
			(g["label"] == first_label).mean()
			if first_label is not None else np.nan
	})

article_eda = (
	dup_df
	.groupby("article", sort=False)
	.apply(analyze_article)
	.reset_index(drop=True)
)

print("\n=== ARTICLE DUPLICATE EDA (HEAD) ===")
print(article_eda.head())

# ================================
# 3. GLOBAL STATS
# ================================
print("\n=== GLOBAL STATS ===")

print("Articles with valid timestamp:",
	  article_eda["has_valid_ts"].mean())

print("Articles with label conflict:",
	  (article_eda["n_labels"] > 1).mean())

print("\nAgreement first-ts vs majority (only valid ts):")
print(
	article_eda.loc[
		article_eda["has_valid_ts"],
		"agree_first_majority"
	].mean()
)

print("\nDeterminism P(label = first_ts_label):")
print(
	article_eda.loc[
		article_eda["has_valid_ts"],
		"p_same_as_first_ts"
	].describe()
)

# ================================
# 4. COMPARE VALID TS vs NO TS
# ================================
print("\n=== COMPARISON VALID TS vs NO TS ===")

compare = article_eda.groupby("has_valid_ts").agg(
	mean_labels=("n_labels", "mean"),
	mean_determinism=("p_same_as_first_ts", "mean"),
	n_articles=("n_rows", "count")
)

print(compare)

# ================================
# 5. WHERE TIMESTAMP DISAGREES
# ================================
disagree = article_eda[
	(article_eda["has_valid_ts"]) &
	(article_eda["agree_first_majority"] == False)
]

print("\nArticles where first timestamp ≠ majority label:",
	  len(disagree))

print(disagree[[
	"labels",
	"first_ts_label",
	"majority_label",
	"p_same_as_first_ts"
]].head(10))

# ================================
# 6. FINAL SUMMARY
# ================================
summary = {
	"dup_articles": article_eda.shape[0],
	"with_valid_ts": article_eda["has_valid_ts"].mean(),
	"label_conflict_rate": (article_eda["n_labels"] > 1).mean(),
	"agreement_first_majority":
		article_eda.loc[
			article_eda["has_valid_ts"],
			"agree_first_majority"
		].mean(),
	"determinism_first_ts":
		article_eda.loc[
			article_eda["has_valid_ts"],
			"p_same_as_first_ts"
		].mean()
}

print("\n=== FINAL SUMMARY ===")
for k, v in summary.items():
	print(f"{k}: {v:.4f}")


Total duplicated rows: 8426
Unique duplicated articles: 2823

=== ARTICLE DUPLICATE EDA (HEAD) ===
   n_rows  n_labels  labels  has_valid_ts  first_ts_label  majority_label  \
0       4         1     [0]          True             0.0               0   
1       2         1     [4]          True             4.0               4   
2       2         2  [3, 5]          True             3.0               3   
3       2         2  [3, 6]         False             NaN               3   
4       5         2  [2, 6]          True             6.0               6   

  agree_first_majority  p_same_as_first_ts  
0                 True                 1.0  
1                 True                 1.0  
2                 True                 0.5  
3                  NaN                 NaN  
4                 True                 0.8  

=== GLOBAL STATS ===
Articles with valid timestamp: 0.7343251859723698
Articles with label conflict: 0.5717321997874601

Agreement first-ts vs majority (only valid ts)

C:\Users\msist\AppData\Local\Temp\ipykernel_51412\2614752166.py:58: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(analyze_article)


In [36]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# ============================================================
# CONFIG
# ============================================================

DEV_PATH = "../data/raw/development.csv"

C_VALUE = 1.5
N_SPLITS = 5
RANDOM_STATE = 42

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]

# ============================================================
# LOAD + BASIC PREP
# ============================================================

df = pd.read_csv(DEV_PATH)

for col in ["article", "title", "source", "timestamp"]:
	df[col] = df[col].fillna("").astype(str)

# valid timestamp (0000-00-00 00:00:00 = missing)
df["has_valid_ts"] = df["timestamp"].ne("0000-00-00 00:00:00")

# text
df["text"] = (df["title"] + " " + df["article"]).str.lower()

# numeric
df["n_tokens"]    = df["article"].str.split().str.len()
df["title_len"]   = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

# ============================================================
# STAGE 1 — TEMPORAL DUPLICATE RULE
# ============================================================

def build_first_timestamp_label_map(df):
	dup = df[df.duplicated("article", keep=False) & df["has_valid_ts"]].copy()
	dup["timestamp_dt"] = pd.to_datetime(dup["timestamp"], errors="coerce")

	first_labels = (
		dup.sort_values("timestamp_dt")
		   .groupby("article")
		   .first()["label"]
	)

	return first_labels.to_dict()

FIRST_TS_LABEL = build_first_timestamp_label_map(df)

def apply_stage1(df):
	mask = df["article"].isin(FIRST_TS_LABEL)
	pred = pd.Series(index=df.index, dtype="float")
	pred[mask] = df.loc[mask, "article"].map(FIRST_TS_LABEL)
	return pred, mask

# ============================================================
# STAGE 2 — BASELINE MODEL
# ============================================================

def make_baseline():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w", TfidfVectorizer(
				ngram_range=(1, 2),
				min_df=2,
				max_df=0.9,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, 5),
				min_df=2,
				max_df=0.9,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([("pre", pre), ("clf", clf)])

# ============================================================
# TWO-STAGE CV EVALUATION
# ============================================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

all_true = []
all_pred = []

for fold, (tr, va) in enumerate(skf.split(df, df["label"]), 1):
	print(f"\n===== FOLD {fold} =====")

	df_tr = df.iloc[tr]
	df_va = df.iloc[va]

	# --- Stage 1
	stage1_pred, stage1_mask = apply_stage1(df_va)

	# --- Stage 2 (only unresolved)
	df_tr_stage2 = df_tr[~df_tr["article"].isin(FIRST_TS_LABEL)]
	df_va_stage2 = df_va[~stage1_mask]

	model = make_baseline()
	model.fit(df_tr_stage2, df_tr_stage2["label"])

	stage2_pred = model.predict(df_va_stage2)

	# --- merge predictions
	y_pred = stage1_pred.copy()
	y_pred.loc[df_va_stage2.index] = stage2_pred

	y_true = df_va["label"].values

	print(classification_report(y_true, y_pred, digits=3))
	print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

	all_true.extend(y_true)
	all_pred.extend(y_pred)

# ============================================================
# GLOBAL RESULT
# ============================================================

print("\n===== GLOBAL TWO-STAGE RESULT =====")
print(classification_report(all_true, all_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(all_true, all_pred))



===== FOLD 1 =====
              precision    recall  f1-score   support

           0      0.803     0.705     0.751      4708
           1      0.743     0.739     0.741      2117
           2      0.838     0.827     0.832      2232
           3      0.594     0.585     0.590      1996
           4      0.832     0.911     0.869      1715
           5      0.513     0.570     0.540      2611
           6      0.600     0.787     0.681       621

    accuracy                          0.715     16000
   macro avg      0.703     0.732     0.715     16000
weighted avg      0.722     0.715     0.716     16000

Confusion Matrix:
 [[3320  154   95  280   43  727   89]
 [  71 1565   90   64   16  270   41]
 [  68  135 1845   66    8   57   53]
 [ 177  100  102 1168  123  265   61]
 [  11   12    2   66 1562   58    4]
 [ 453  126   57  290  120 1487   78]
 [  32   15   11   31    6   37  489]]

===== FOLD 2 =====
              precision    recall  f1-score   support

           0      0.80